# 01a_streaming_filter: Polars scan + streaming collect

Goal: generate a realistic (but synthetic) wearable dataset and use Polars *lazy scans* to filter and aggregate without ever materializing the full table in memory.

This demo is intentionally boring-data-science: scan → filter → select → group_by → collect(streaming=True) → write Parquet.

## Prereqs

- From repo root: `uv pip install -r requirements.txt`

## Data setup

From `lectures/02/demo/`:

In [1]:
%%bash
mkdir -p data outputs
uv run python generate_demo_data.py --size small --output-dir data

2026-01-13 05:57:34,661 - INFO - Generating small dataset:
2026-01-13 05:57:34,661 - INFO -   - User

s: 150


2026-01-13 05:57:34,661 - INFO -   - Duration: 35 days
2026-01-13 05:57:34,661 - INFO -   - Target s

ensor rows: 1,000,000


2026-01-13 05:57:34,661 - INFO - Generating user profiles + sleep diaries...


...

2026-01-13 05:57:35,241 - INFO - Generation complete!


2026-01-13 05:57:35,241 - INFO - Files written to: data
2026-01-13 05:57:35,241 - INFO -  - sensor_h

rv/: 1,000,000 rows, 38.2 MB
2026-01-13 05:57:35,241 - INFO -  - sleep_diary.parquet: 5,250 rows, 0.

1 MB
2026-01-13 05:57:35,241 - INFO -  - user_profile.parquet: 150 rows, 0.0 MB


2026-01-13 05:57:35,243 - INFO - Metadata written to: data/generation_metadata.yaml


You should now have:

- `data/sensor_hrv/part-*.parquet`
- `data/sleep_diary.parquet`
- `data/user_profile.parquet`

## Steps

In [2]:
from pathlib import Path

sensor_dir = Path("data/sensor_hrv")
required_files = [
    Path("data/sleep_diary.parquet"),
    Path("data/user_profile.parquet"),
]

missing_files = [p for p in required_files if not p.exists()]
sensor_parts = list(sensor_dir.glob("*.parquet"))

if missing_files or not sensor_parts:
    raise FileNotFoundError(
        "Missing demo data.\n"
        f"- Missing files: {', '.join(str(p) for p in missing_files) if missing_files else '(none)'}\n"
        f"- Sensor parts found: {len(sensor_parts)}\n"
        "Run: uv run python generate_demo_data.py --size small --output-dir data"
    )

### 1) Scan the big table lazily

In [3]:
import polars as pl

sensor = pl.scan_parquet("data/sensor_hrv/*.parquet")
print(sensor.schema)

Schema({'device_id': String, 'ts_start': Datetime(time_unit='us', time_zone=None), 'ts_end': Datetime(time_unit='us', time_zone=None), 'heart_rate': Int64, 'hrv_sdnn': Float64, 'hrv_rmssd': Float64, 'hrv_lf_hf_ratio': Float64, 'bvp_mean': Float64, 'spo2': Float64, 'eda_mean': Float64, 'skin_temp': Float64, 'acc_magnitude': Float64, 'steps': Int64, 'missingness_score': Float64})


/tmp/ipykernel_238398/1256061096.py:4: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(sensor.schema)


Key point: `scan_parquet` builds a query plan; it does not load the full table.

### 2) Filter + project early (predicate/projection pushdown)

In [4]:
import polars as pl

query = (
    pl.scan_parquet("data/sensor_hrv/*.parquet")
    .filter(pl.col("missingness_score") <= 0.35)
    .filter(pl.col("ts_start") >= pl.datetime(2024, 1, 15))
    .select([
        "device_id",
        pl.col("ts_start").dt.date().alias("date"),
        "heart_rate",
        "hrv_sdnn",
        "steps",
    ])
)

print(query.explain())

SELECT [col("device_id"), col("ts_start").dt.date().alias("date"), col("heart_rate"), col("hrv_sdnn"), col("steps")]
  Parquet SCAN [data/sensor_hrv/part-00000.parquet, ... 4 other sources]
  PROJECT 6/14 COLUMNS
  SELECTION: [([(col("ts_start")) >= (2024-01-15 00:00:00)]) & ([(col("missingness_score")) <= (0.35)])]
  ESTIMATED ROWS: 1000000


### 3) Aggregate to a daily summary and stream the collect

In [5]:
import polars as pl
from pathlib import Path

daily = (
    query
    .group_by(["device_id", "date"])
    .agg([
        pl.len().alias("num_segments"),
        pl.mean("heart_rate").alias("mean_hr"),
        pl.mean("hrv_sdnn").alias("mean_sdnn"),
        pl.sum("steps").alias("steps_sum"),
    ])
    .sort(["device_id", "date"])
)

result = daily.collect(engine="streaming")
Path("outputs").mkdir(parents=True, exist_ok=True)
result.write_parquet("outputs/daily_device_summary.parquet")
print(result.head())

shape: (5, 6)
┌───────────┬────────────┬──────────────┬───────────┬───────────┬───────────┐
│ device_id ┆ date       ┆ num_segments ┆ mean_hr   ┆ mean_sdnn ┆ steps_sum │
│ ---       ┆ ---        ┆ ---          ┆ ---       ┆ ---       ┆ ---       │
│ str       ┆ date       ┆ u32          ┆ f64       ┆ f64       ┆ i64       │
╞═══════════╪════════════╪══════════════╪═══════════╪═══════════╪═══════════╡
│ DEV-00000 ┆ 2024-01-15 ┆ 117          ┆ 83.863248 ┆ 82.291538 ┆ 3946      │
│ DEV-00000 ┆ 2024-01-16 ┆ 132          ┆ 87.113636 ┆ 80.564394 ┆ 4841      │
│ DEV-00000 ┆ 2024-01-17 ┆ 134          ┆ 87.074627 ┆ 84.007612 ┆ 4371      │
│ DEV-00000 ┆ 2024-01-18 ┆ 110          ┆ 84.363636 ┆ 80.710091 ┆ 3868      │
│ DEV-00000 ┆ 2024-01-19 ┆ 154          ┆ 87.207792 ┆ 82.148377 ┆ 5005      │
└───────────┴────────────┴──────────────┴───────────┴───────────┴───────────┘


## Checkpoints

- `outputs/daily_device_summary.parquet` exists
- `result.height > 0`
- `num_segments` looks like "a day of 5-min windows" (usually a couple hundred)

## Stretch (optional)

- Change the filter to keep only nighttime segments and compare `mean_sdnn`.
- Try `.collect()` without `streaming=True` and watch memory/time differences on `--size large`.